In [1]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import joblib
import os
# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_CLASSES = 7  # Modify according to your dataset

In [4]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from tqdm import tqdm
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7

# --- Model with Dropout ---
class ResNetWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.base_model = models.resnet18(pretrained=False)
        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, NUM_CLASSES)
        )

    def forward(self, x):
        return self.base_model(x)

# Instantiate and load model
model = ResNetWithDropout(dropout_rate=0.3)
model_path = os.path.join("models", "best_model_resnet.pth")
state_dict = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state_dict)  # ✅ Do not modify keys
model.to(DEVICE).eval()

# ---------------- FEATURE EXTRACTOR ----------------
feature_extractor = nn.Sequential(
    *list(model.base_model.children())[:-1]  # Remove Dropout + Linear
)
feature_extractor.to(DEVICE).eval()

# ---------------- FEATURE EXTRACTION ----------------
def extract_features(dataloader, extractor):
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting features"):
            imgs = imgs.to(DEVICE)
            feats = extractor(imgs)
            feats = feats.view(feats.size(0), -1)  # Flatten
            features.append(feats.cpu().numpy())
            labels.extend(lbls.numpy())
    return np.vstack(features), np.array(labels)


c:\Users\mohamed ahmed\anaconda3\envs\tfenv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\mohamed ahmed\anaconda3\envs\tfenv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
import cv2
from torchvision import models, transforms
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 7  # You can adjust this depending on your labels

# --- Label mapping ---
label_map = {
    'angry': 0,
    'disgust': 1,
    'fear': 2,
    'happy': 3,
    'sad': 4,
    'surprise': 5,
    'neutral': 6
}

# Load images from folder
def load_images_from_folder(folder_path, img_size=(48, 48)):
    X, y = [], []
    for label in os.listdir(folder_path):  # e.g., "angry", "happy", etc.
        label_path = os.path.join(folder_path, label)
        if not os.path.isdir(label_path):
            continue
        for img_file in os.listdir(label_path):
            img_path = os.path.join(label_path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, img_size)
                X.append(img)
                y.append(label_map[label.lower()])  # map string to integer label
    return np.array(X), np.array(y)

# Load train and test
current_directory = os.getcwd()
train_path = os.path.join(current_directory, 'train')
test_path = os.path.join(current_directory, 'test')
X_train, y_train = load_images_from_folder(train_path)
X_test, y_test = load_images_from_folder(test_path)

# Normalize pixel values
X_train = X_train / 255.0
X_test = X_test / 255.0

# Reshape for CNN input: (samples, height, width, channels)
X_train = X_train.reshape(-1, 48, 48, 1)
X_test = X_test.reshape(-1, 48, 48, 1)

# One-hot encode test set only (not train yet)
y_test = to_categorical(y_test, num_classes=7)

# Desired validation class counts (same as test)
desired_val_counts = {
    0: 958,   # Angry
    1: 111,   # Disgust
    2: 1024,  # Fear
    3: 1774,  # Happy
    4: 1247,  # Sad
    5: 831,   # Surprise
    6: 1233   # Neutral
}

# Convert X_train to numpy array (in case it's a list)
X_train = np.array(X_train)
y_train = np.array(y_train)

X_val, y_val = [], []
used_indices = set()

# Extract validation samples matching test distribution
for label, count in desired_val_counts.items():
    indices = np.where(y_train == label)[0]
    selected = np.random.choice(indices, size=count, replace=False)
    X_val.append(X_train[selected])
    y_val.append(y_train[selected])
    used_indices.update(selected.tolist())

# Combine validation
X_val = np.concatenate(X_val)
y_val = np.concatenate(y_val)

# Build the final training set with the rest
mask = np.ones(len(X_train), dtype=bool)
mask[list(used_indices)] = False
X_train_final = X_train[mask]
y_train_final = y_train[mask]

# Shuffle final train set
X_train_final, y_train_final = shuffle(X_train_final, y_train_final, random_state=42)

# One-hot encode labels
y_train_final_oh = to_categorical(y_train_final, num_classes=7)
y_val_oh = to_categorical(y_val, num_classes=7)

# Reverse label map for plotting
label_map_inv = {v: k.capitalize() for k, v in label_map.items()}

# For visualization and counts
y_train_labels = y_train_final
y_val_labels = y_val
y_test_labels = np.argmax(y_test, axis=1)

# Print class counts
def print_class_counts(name, labels):
    unique_labels, counts = np.unique(labels, return_counts=True)
    print(f"\n{name} Class Counts:")
    for label, count in zip(unique_labels, counts):
        print(f"{label_map_inv[label]}: {count}")

print_class_counts("Train", y_train_labels)
print_class_counts("Test", y_test_labels)
print_class_counts("Validation", y_val_labels)

# --- Dataset Class ---
class FERDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        if image.ndim == 3:
            image = image.squeeze()

        if image.max() <= 1.0:
            image = (image * 255).astype(np.uint8)

        image = Image.fromarray(image).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# --- Transforms ---
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- Dataset and DataLoaders ---
batch_size = 128

# Convert one-hot labels to class indices if needed
if y_test.ndim > 1 and y_test.shape[1] > 1:
    y_test = np.argmax(y_test, axis=1)
if y_val.ndim > 1 and y_val.shape[1] > 1:
    y_val = np.argmax(y_val, axis=1)
if y_train_final.ndim > 1 and y_train_final.shape[1] > 1:
    y_train_final = np.argmax(y_train_final, axis=1)

train_dataset = FERDataset(X_train_final, y_train_final, train_transform)
val_dataset = FERDataset(X_val, y_val, eval_transform)
test_dataset = FERDataset(X_test, y_test, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Now the data is prepared and ready for training




Train Class Counts:
Angry: 3037
Disgust: 325
Fear: 3073
Happy: 5441
Sad: 3583
Surprise: 2340
Neutral: 3732

Test Class Counts:
Angry: 958
Disgust: 111
Fear: 1024
Happy: 1774
Sad: 1247
Surprise: 831
Neutral: 1233

Validation Class Counts:
Angry: 958
Disgust: 111
Fear: 1024
Happy: 1774
Sad: 1247
Surprise: 831
Neutral: 1233


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- EXTRACT FEATURES ----------------
# Assuming you already have these functions
X_train, y_train = extract_features(train_loader, feature_extractor)
X_val, y_val = extract_features(val_loader, feature_extractor)
X_test, y_test = extract_features(test_loader, feature_extractor)

# ---------------- SCALING ----------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ---------------- OPTIONAL: PCA ----------------
# Retain 95% of variance to reduce dimensionality
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

# ---------------- GRID SEARCH FOR BEST C ----------------
param_grid = {"C": [0.001, 0.01, 0.1, 1.0, 10.0]}
grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid.fit(X_train_pca, y_train)

print(f"✅ Best C found: {grid.best_params_['C']}")

# ---------------- BEST MODEL ----------------
best_logreg = grid.best_estimator_

# ---------------- VALIDATION PERFORMANCE ----------------
y_pred_val = best_logreg.predict(X_val_pca)
val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"✅ Validation Accuracy: {val_accuracy:.4f}")
print("Validation Classification Report:")
print(classification_report(y_val, y_pred_val))

Extracting features:   5%|▌         | 9/169 [00:51<14:49,  5.56s/it]

In [ ]:

# ---------------- TEST PERFORMANCE ----------------
y_pred_test = best_logreg.predict(X_test_pca)
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"✅ Test Accuracy: {test_accuracy:.4f}")
print("Test Classification Report:")
print(classification_report(y_test, y_pred_test))

# ---------------- CONFUSION MATRIX ----------------
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test, y_pred_test, title="Test Set Confusion Matrix")

# ---------------- SAVE MODEL, SCALER, PCA ----------------
joblib.dump(best_logreg, "logistic_cnn_with_pca_grid.pkl")
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(pca, "pca_transform.pkl")
print("✅ Saved Logistic Regression model, Scaler, and PCA")


Extracting features: 100%|██████████| 57/57 [03:53<00:00,  4.10s/it]

✅ Example Prediction (first 10): [0 0 4 4 0 4 6 0 0 0]
